# Linear Regression: Predict Orders for the Next 10 Weeks

This notebook fits a linear regression model on `train.csv` and predicts `num_orders` for every center-meal row in weeks 146–155 of `test.csv`. It checks performance on the last 10 labeled weeks first. The source CSV files are read-only; predictions are saved as a separate derived file.

In [1]:
# Import the tools for reading CSV files, preparing model inputs, and evaluating predictions.
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Find the project root whether this notebook is launched from the root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "data" / "raw" / "train.csv").exists():
    project_root = project_root.parent

# Read the labeled training set and the future test set without editing either source file.
train_data = pd.read_csv(project_root / "data" / "raw" / "train.csv")
test_data = pd.read_csv(project_root / "data" / "raw" / "test.csv")

# Confirm the row counts and week coverage before preparing the model.
print("Training rows and columns:", train_data.shape)
print("Test rows and columns:", test_data.shape)
print("Training weeks:", train_data["week"].min(), "to", train_data["week"].max())
print("Test weeks:", test_data["week"].min(), "to", test_data["week"].max())

Training rows and columns: (456548, 9)
Test rows and columns: (32573, 8)
Training weeks: 1 to 145
Test weeks: 146 to 155


In [2]:
# Keep only predictors available in both datasets; num_orders is the training target.
feature_columns = [
    "week",
    "center_id",
    "meal_id",
    "checkout_price",
    "base_price",
    "emailer_for_promotion",
    "homepage_featured",
]
categorical_columns = ["center_id", "meal_id"]

# Drop rows missing required predictors or the training target, and report any rows removed.
train_rows_before = len(train_data)
test_rows_before = len(test_data)
train_data = train_data.dropna(subset=feature_columns + ["num_orders"]).reset_index(drop=True)
test_data = test_data.dropna(subset=feature_columns).reset_index(drop=True)
print("Training rows after drop:", len(train_data), "(removed", train_rows_before - len(train_data), ")")
print("Test rows after drop:", len(test_data), "(removed", test_rows_before - len(test_data), ")")

# Separate the predictor fields from the target order counts.
train_features = train_data[feature_columns].copy()
test_features = test_data[feature_columns].copy()
for column in categorical_columns:
    train_features[column] = train_features[column].astype(str)
    test_features[column] = test_features[column].astype(str)
train_target = train_data["num_orders"]

# One-hot encode center and meal IDs; pass numeric fields through unchanged.
# Sparse encoding avoids creating a large dense matrix for all training rows.
preprocessor = ColumnTransformer(
    transformers=[("categories", OneHotEncoder(handle_unknown="ignore"), categorical_columns)],
    remainder="passthrough",
)

# Combine preprocessing with ordinary least-squares linear regression in one reusable pipeline.
linear_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regression", LinearRegression()),
    ]
)
print("Features:", feature_columns)
print("Target: num_orders")

Training rows after drop: 456548 (removed 0 )
Test rows after drop: 32573 (removed 0 )


Features:

 ['week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured']
Target: num_orders


In [3]:
# Reserve the latest 10 labeled weeks as a chronological validation set.
validation_start_week = int(train_data["week"].max()) - 9
fit_mask = train_data["week"] < validation_start_week
validation_mask = train_data["week"] >= validation_start_week

# Fit the linear model on earlier weeks, then predict the held-out later weeks.
linear_model.fit(train_features.loc[fit_mask], train_target.loc[fit_mask])
validation_predictions = linear_model.predict(train_features.loc[validation_mask])

# Measure typical absolute error and penalize larger errors with root mean squared error.
validation_actual = train_target.loc[validation_mask]
mae = mean_absolute_error(validation_actual, validation_predictions)
rmse = np.sqrt(mean_squared_error(validation_actual, validation_predictions))
print("Validation weeks:", validation_start_week, "to", int(train_data["week"].max()))
print("Validation rows:", int(validation_mask.sum()))
print(f"Mean absolute error (orders): {mae:,.2f}")
print(f"Root mean squared error (orders): {rmse:,.2f}")

Validation weeks: 136 to 145
Validation rows: 32821
Mean absolute error (orders): 136.38
Root mean squared error (orders): 191.66


In [ ]:
# Refit the same model on all labeled training weeks before predicting the future test weeks.
linear_model.fit(train_features, train_target)
test_predictions = linear_model.predict(test_features)

# Order counts cannot be negative, so floor any negative estimates at zero.
predictions = test_data[["id", "week"]].copy()
predictions["predicted_num_orders"] = np.maximum(test_predictions, 0)

# Save one prediction per test row outside the raw-data folder and display a sample.
output_directory = project_root / "data" / "processed"
output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / "linear_regression_predictions.csv"
predictions.to_csv(output_path, index=False)
print("Predicted test rows:", len(predictions))
print("Predicted weeks:", predictions["week"].min(), "to", predictions["week"].max())
print("Prediction file:", output_path)
predictions.head()